# Food Delivery Analytics — Phase 1: Preprocessing & Feature Engineering

**Project:** Food Delivery Analytics: Delivery Performance and Delay Prediction  
**Dataset:** `data/raw/Order_delivery.csv`  
**Phase scope:** Dataset profiling · Preprocessing · Feature engineering · Quality validation

---

> **Reproducibility note:**  
> This notebook is an interactive walkthrough. The production scripts are:  
> - `src/preprocessing/preprocess.py` — run with `python src/preprocessing/preprocess.py`  
> - `src/preprocessing/feature_engineering.py` — run with `python src/preprocessing/feature_engineering.py`

---
## 0 · Setup

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Resolve paths relative to project root (works whether notebook is in notebooks/ or root)
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = NOTEBOOK_DIR if os.path.exists(os.path.join(NOTEBOOK_DIR, 'data')) \
               else os.path.dirname(NOTEBOOK_DIR)

RAW_PATH      = os.path.join(PROJECT_ROOT, 'data', 'raw',       'Order_delivery.csv')
CLEANED_PATH  = os.path.join(PROJECT_ROOT, 'data', 'processed', 'Order_delivery_cleaned.csv')
FEATURED_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'Order_delivery_featured.csv')

print('Project root :', PROJECT_ROOT)
print('Pandas       :', pd.__version__)
print('NumPy        :', np.__version__)

---
## 1 · Load Raw Dataset

In [ ]:
raw = pd.read_csv(RAW_PATH)
print(f'Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
raw.head(3)

---
## 2 · Dataset Profiling

In [ ]:
# Column names, dtypes, and null counts
profile = pd.DataFrame({
    'dtype'  : raw.dtypes,
    'null'   : raw.isnull().sum(),
    'nunique': raw.nunique(),
    'sample' : [raw[c].dropna().iloc[0] if len(raw[c].dropna()) > 0 else None for c in raw.columns]
})
profile

In [ ]:
# Numerical summary
raw.describe()

In [ ]:
# Categorical value counts
cat_cols = ['City','Item_Name','Payment_Method','Order_Status',
            'Driver_Vehicle','Traffic_Level','Driver_Availability']
for col in cat_cols:
    vc = raw[col].value_counts()
    pct = (vc / len(raw) * 100).round(2)
    print(f'\n--- {col} ({raw[col].nunique()} unique) ---')
    print(pd.DataFrame({'count': vc, 'pct': pct}))

In [ ]:
# Full-row and Order_ID duplicates
print('Full-row duplicates:', raw.duplicated().sum())
print('Duplicate Order_IDs:', raw['Order_ID'].duplicated().sum())
print('Total missing cells:', raw.isnull().sum().sum())

In [ ]:
# Datetime parsing check
ot = pd.to_datetime(raw['Order_Time'],    format='%d-%m-%Y %H:%M', errors='coerce')
dt = pd.to_datetime(raw['Delivery_Time'], format='%d-%m-%Y %H:%M', errors='coerce')
print('Order_Time NaT   :', ot.isna().sum())
print('Delivery_Time NaT:', dt.isna().sum())
print('Order_Time range :', ot.min(), '->', ot.max())
print('Delivery < Order :', (dt < ot).sum())
computed = (dt - ot).dt.total_seconds() / 60
print('Duration mismatch (>1 min):', (computed - raw['Delivery_Duration_Minutes']).abs().gt(1).sum())

---
## 3 · Run Preprocessing Pipeline

In [ ]:
# Run the production script
import subprocess
result = subprocess.run(
    [sys.executable, os.path.join(PROJECT_ROOT, 'src', 'preprocessing', 'preprocess.py')],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

---
## 4 · Inspect Cleaned Dataset

In [ ]:
cleaned = pd.read_csv(CLEANED_PATH)
print(f'Cleaned shape: {cleaned.shape[0]:,} rows x {cleaned.shape[1]} columns')
print(f'New columns  : {[c for c in cleaned.columns if c not in raw.columns]}')
print(f'Missing      : {cleaned.isnull().sum().sum()}')
print(f'Duplicates   : {cleaned.duplicated().sum()}')
cleaned.head(3)

In [ ]:
# Verify datetime parsing
cleaned['Order_Time']    = pd.to_datetime(cleaned['Order_Time'],    format='%Y-%m-%d %H:%M')
cleaned['Delivery_Time'] = pd.to_datetime(cleaned['Delivery_Time'], format='%Y-%m-%d %H:%M')
print('Order_Time dtype   :', cleaned['Order_Time'].dtype)
print('Delivery_Time dtype:', cleaned['Delivery_Time'].dtype)

# Status flag check
print('\nStatus flags (sum should equal value_counts):')
print('Is_Delivered :', cleaned['Is_Delivered'].sum())
print('Is_Cancelled :', cleaned['Is_Cancelled'].sum())
print('Is_InTransit :', cleaned['Is_InTransit'].sum())
print('Total         :', cleaned[['Is_Delivered','Is_Cancelled','Is_InTransit']].sum().sum())

---
## 5 · Run Feature Engineering Pipeline

In [ ]:
result2 = subprocess.run(
    [sys.executable, os.path.join(PROJECT_ROOT, 'src', 'preprocessing', 'feature_engineering.py')],
    capture_output=True, text=True
)
print(result2.stdout[-3000:] if len(result2.stdout) > 3000 else result2.stdout)
if result2.returncode != 0:
    print('STDERR:', result2.stderr)

---
## 6 · Inspect Featured Dataset

In [ ]:
featured = pd.read_csv(FEATURED_PATH)
print(f'Featured shape: {featured.shape[0]:,} rows x {featured.shape[1]} columns')
fe_new = [c for c in featured.columns if c not in cleaned.columns]
print(f'New FE columns ({len(fe_new)}): {fe_new}')
featured.head(3)

In [ ]:
# Target variable summary
delivered = featured[featured['Order_Status'] == 'Delivered']
n_delayed = delivered['Is_Delayed_TARGET'].sum()
n_total   = len(delivered)
print(f'Delay threshold        : 45 minutes (p75 of Delivered orders)')
print(f'Delivered orders       : {n_total:,}')
print(f'Delayed (> 45 min)     : {n_delayed:,}  ({n_delayed/n_total*100:.1f}%)')
print(f'On-time (<= 45 min)    : {n_total-n_delayed:,}  ({(n_total-n_delayed)/n_total*100:.1f}%)')
print()
print('Duration_Bin_TARGET distribution (Delivered):')
print(delivered['Duration_Bin_TARGET'].value_counts().sort_index())

In [ ]:
# Safe feature list (PRE_ORDER, no leakage)
LEAKAGE_COLS = {
    'Delivery_Time','Delivery_Duration_Minutes','Order_Status',
    'Is_Delivered','Is_Cancelled','Is_InTransit',
    'Is_Delayed_TARGET','Delay_Minutes_TARGET','Duration_Bin_TARGET',
    'Order_To_Delivery_Minutes_TARGET',
    'Delivery_Speed_kmh_ANALYTICAL','Distance_Per_Minute_ANALYTICAL',
}
ID_COLS = {
    'Order_ID','User_ID','Restaurant_ID','Driver_ID',
    'Order_Time','Order_Date','Order_DayName',
}
ZERO_VAR = {
    'Restaurant_Geo_Imputed','Customer_Geo_Imputed','Driver_Geo_Imputed',
    'Same_Location_Flag','Order_Month',
}

safe_features = [
    c for c in featured.columns
    if c not in LEAKAGE_COLS and c not in ID_COLS and c not in ZERO_VAR
]
print(f'Safe PRE_ORDER input features ({len(safe_features)}):')
for f in safe_features:
    print(f'  {f}')

---
## 7 · Phase 1 Summary

| Item | Value |
|---|---|
| Raw rows | 100,002 |
| Cleaned rows | 100,002 (0 removed) |
| Raw columns | 23 |
| Cleaned columns | 38 (+15) |
| Featured columns | 55 (+17) |
| Missing values | 0 |
| Duplicates | 0 |
| Validation checks | 26 PASS · 7 WARN · 0 FAIL |
| Delay threshold | 45 min (p75 of Delivered orders) |
| Delayed orders | 20,675 / 85,199 Delivered (24.3%) |

### Phase 1 is complete. Next: Phase 2 — Exploratory Data Analysis.